<a href="https://colab.research.google.com/github/umichael779/pytorch_wine_classification_model/blob/main/pytorch_wine_classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [2]:
torch.manual_seed(42)

In [3]:
data = load_wine()

In [4]:
data.data

array([[1.423e+01, 1.710e+00, 2.430e+00, ..., 1.040e+00, 3.920e+00,
        1.065e+03],
       [1.320e+01, 1.780e+00, 2.140e+00, ..., 1.050e+00, 3.400e+00,
        1.050e+03],
       [1.316e+01, 2.360e+00, 2.670e+00, ..., 1.030e+00, 3.170e+00,
        1.185e+03],
       ...,
       [1.327e+01, 4.280e+00, 2.260e+00, ..., 5.900e-01, 1.560e+00,
        8.350e+02],
       [1.317e+01, 2.590e+00, 2.370e+00, ..., 6.000e-01, 1.620e+00,
        8.400e+02],
       [1.413e+01, 4.100e+00, 2.740e+00, ..., 6.100e-01, 1.600e+00,
        5.600e+02]])

In [5]:
data = load_wine(as_frame=True)

In [6]:
df = data.frame

In [7]:
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [8]:
df.shape

(178, 14)

In [9]:
df.describe()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
count,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000
mean,13.000618,2.336348,2.366517,19.494944,99.741573,2.295112,2.029270,0.361854,1.590899,5.058090,0.957449,2.611685,746.893258,0.938202
std,0.811827,1.117146,0.274344,3.339564,14.282484,0.625851,0.998859,0.124453,0.572359,2.318286,0.228572,0.709990,314.907474,0.775035
min,11.030000,0.740000,1.360000,10.600000,70.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000,0.000000
25%,12.362500,1.602500,2.210000,17.200000,88.000000,1.742500,1.205000,0.270000,1.250000,3.220000,0.782500,1.937500,500.500000,0.000000
50%,13.050000,1.865000,2.360000,19.500000,98.000000,2.355000,2.135000,0.340000,1.555000,4.690000,0.965000,2.780000,673.500000,1.000000
75%,13.677500,3.082500,2.557500,21.500000,107.000000,2.800000,2.875000,0.437500,1.950000,6.200000,1.120000,3.170000,985.000000,2.000000
max,14.830000,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,1680.000000,2.000000


In [22]:
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.2, random_state=42, stratify=data.target)


In [23]:
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

In [28]:
X_tr = torch.tensor(np.asarray(X_tr), dtype=torch.float32)
X_te = torch.tensor(np.asarray(X_te), dtype=torch.float32)
y_tr = torch.tensor(np.asarray(y_tr), dtype=torch.long)
y_te = torch.tensor(np.asarray(y_te), dtype=torch.long)

In [29]:
train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)

In [30]:
class Net(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.net(x)

In [31]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Net(X_tr.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [35]:
for epoch in range(60):
    model.train()
    running = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        # Squeeze yb to make it a 1D tensor of shape [batch_size]
        loss = criterion(model(xb), yb.squeeze())
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f"Epoch {epoch+1:3d}, Loss: {running/len(train_dl):.4f}")

Epoch   1, Loss: 1.1134
Epoch   2, Loss: 1.0987
Epoch   3, Loss: 1.0844
Epoch   4, Loss: 1.0789
Epoch   5, Loss: 1.0631
Epoch   6, Loss: 1.0478
Epoch   7, Loss: 1.0413
Epoch   8, Loss: 1.0169
Epoch   9, Loss: 1.0110
Epoch  10, Loss: 0.9807
Epoch  11, Loss: 0.9575
Epoch  12, Loss: 0.9280
Epoch  13, Loss: 0.8949
Epoch  14, Loss: 0.8682
Epoch  15, Loss: 0.8420
Epoch  16, Loss: 0.7892
Epoch  17, Loss: 0.7679
Epoch  18, Loss: 0.7198
Epoch  19, Loss: 0.6762
Epoch  20, Loss: 0.6198
Epoch  21, Loss: 0.5744
Epoch  22, Loss: 0.5305
Epoch  23, Loss: 0.5116
Epoch  24, Loss: 0.4740
Epoch  25, Loss: 0.4338
Epoch  26, Loss: 0.3836
Epoch  27, Loss: 0.4095
Epoch  28, Loss: 0.3709
Epoch  29, Loss: 0.3956
Epoch  30, Loss: 0.2844
Epoch  31, Loss: 0.2784
Epoch  32, Loss: 0.3020
Epoch  33, Loss: 0.2613
Epoch  34, Loss: 0.2321
Epoch  35, Loss: 0.2243
Epoch  36, Loss: 0.2210
Epoch  37, Loss: 0.1928
Epoch  38, Loss: 0.1919
Epoch  39, Loss: 0.1799
Epoch  40, Loss: 0.1751
Epoch  41, Loss: 0.1453
Epoch  42, Loss:

In [37]:
model.eval()
with torch.no_grad():
    pred = model(X_te.to(device))
    probs = torch.softmax(pred, dim=1).cpu()
    preds = torch.argmax(pred, dim=1).cpu()
    acc = (preds == y_te.squeeze()).float().mean().item()
print(f"Accuracy: {acc:.2%}")

Accuracy: 94.44%


In [38]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_te, preds))
print(classification_report(y_te, preds, target_names=data.target_names))

[[12  0  0]
 [ 1 13  0]
 [ 0  1  9]]
              precision    recall  f1-score   support

     class_0       0.92      1.00      0.96        12
     class_1       0.93      0.93      0.93        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.94        36
   macro avg       0.95      0.94      0.95        36
weighted avg       0.95      0.94      0.94        36

